In [1]:
from pyspark.sql import SparkSession

In [2]:
spark_job = SparkSession.builder.appName('NYSC taxi Project').config("spark.driver.memory", "1500m").getOrCreate()
spark_job

c:\Users\user\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


### DATA INGESTION

In [3]:
df = spark_job.read.option('header', 'true').csv('C:/Users/user/Desktop/Data Engineering/Yellow Trip ETL/data/*.csv', inferSchema=True)
df.show()

+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|   pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag|  dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+-------------------+------------------+----------+------------------+-------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       2| 2015-01-15 19:05:39|  2015-01-15 19:23:42|              1|         1.59|   -73.993896484375|40.750110626220703|        

In [4]:
print(spark_job.sparkContext.getConf().get('spark.driver.memory'))

1500m


### DATA PROFILING

In [5]:
from pyspark.sql import functions as F

df.printSchema()
print(f"Row count: {df.count()}")
print(f"Number of partitions: {df.rdd.getNumPartitions()}")

# null count per column
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show()

# duplicate check
print(f"Distinct rows: {df.distinct().count()}")

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: string (nullable = true)
 |-- tpep_dropoff_datetime: string (nullable = true)
 |-- passenger_count: string (nullable = true)
 |-- trip_distance: string (nullable = true)
 |-- pickup_longitude: string (nullable = true)
 |-- pickup_latitude: string (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: string (nullable = true)
 |-- dropoff_latitude: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: string (nullable = true)
 |-- extra: string (nullable = true)
 |-- mta_tax: string (nullable = true)
 |-- tip_amount: string (nullable = true)
 |-- tolls_amount: string (nullable = true)
 |-- improvement_surcharge: string (nullable = true)
 |-- total_amount: string (nullable = true)

Row count: 23655844
Number of partitions: 28
+--------+--------------------+---------------------+---------------+-------------

Descriptive Statistics

In [6]:
df.describe(
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "tip_amount",
    "total_amount"
).show()

+-------+------------------+-----------------+------------------+-----------------+------------------+
|summary|   passenger_count|    trip_distance|       fare_amount|       tip_amount|      total_amount|
+-------+------------------+-----------------+------------------+-----------------+------------------+
|  count|          23655844|         23655844|          23655844|         23655844|          23655844|
|   mean|1.6765832578199282|9.396725794268752|12.173662116219514|1.806254650225065|15.354088082091266|
| stddev|1.3319409759089829|7504.911428957069|25.306932006369735|812.2589156619586| 812.6852760790412|
|    min|                 0|              .00|             -0.01|            -0.01|             -0.31|
|    max|                 9|            99.90|            999.99|           998.14|             998.3|
+-------+------------------+-----------------+------------------+-----------------+------------------+



### DATA CLEANING

In [7]:
from pyspark.sql import functions as F

df = df.withColumn("tpep_pickup_datetime", F.to_timestamp("tpep_pickup_datetime")) \
    .withColumn("tpep_dropoff_datetime", F.to_timestamp("tpep_dropoff_datetime")) \
    .withColumn("passenger_count", F.col("passenger_count").cast("integer")) \
    .withColumn("trip_distance", F.col("trip_distance").cast("double")) \
    .withColumn("fare_amount", F.col("fare_amount").cast("double")) \
    .withColumn("extra", F.col("extra").cast("double")) \
    .withColumn("mta_tax", F.col("mta_tax").cast("double")) \
    .withColumn("tip_amount", F.col("tip_amount").cast("double")) \
    .withColumn("tolls_amount", F.col("tolls_amount").cast("double")) \
    .withColumn("improvement_surcharge", F.col("improvement_surcharge").cast("double")) \
    .withColumn("total_amount", F.col("total_amount").cast("double")) \
    .withColumn("pickup_longitude", F.col("pickup_longitude").cast("double")) \
    .withColumn("pickup_latitude", F.col("pickup_latitude").cast("double")) \
    .withColumn("dropoff_longitude", F.col("dropoff_longitude").cast("double")) \
    .withColumn("dropoff_latitude", F.col("dropoff_latitude").cast("double"))

df.printSchema()

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [8]:
df_clean = df.filter(
    (F.col("trip_distance") > 0) &
    (F.col("fare_amount") > 0) &
    (F.col("passenger_count") > 0) &
    (F.col("tpep_dropoff_datetime") > F.col("tpep_pickup_datetime"))
)

print(f"Before: {df.count()}, After cleaning: {df_clean.count()}")

Before: 23655844, After cleaning: 23494548


### FEATURE ENGINEERING

In [9]:
df_clean = df_clean.withColumn(
    "trip_duration_min",
    (F.col("tpep_dropoff_datetime").cast("long") - F.col("tpep_pickup_datetime").cast("long")) / 60
).withColumn(
    "pickup_hour", F.hour("tpep_pickup_datetime")
).withColumn(
    "day_of_week", F.date_format("tpep_pickup_datetime", "EEEE")
).withColumn(
    "fare_per_mile", F.when(F.col("trip_distance") > 0, F.col("fare_amount") / F.col("trip_distance"))
).withColumn(
    "tip_pct", F.when(F.col("fare_amount") > 0, (F.col("tip_amount") / F.col("fare_amount")) * 100)
).withColumn("average_speed_mph", F.try_divide(
    F.col("trip_distance"),
    F.col("trip_duration_min") / 60
))

In [10]:
df_clean.select(
    'trip_duration_min',
    'pickup_hour',
    'day_of_week',
    'fare_per_mile',
    'tip_pct',
    'average_speed_mph'
).show()

+------------------+-----------+-----------+------------------+------------------+------------------+
| trip_duration_min|pickup_hour|day_of_week|     fare_per_mile|           tip_pct| average_speed_mph|
+------------------+-----------+-----------+------------------+------------------+------------------+
|             18.05|         19|   Thursday| 7.547169811320754|27.083333333333332| 5.285318559556787|
|19.833333333333332|         20|   Saturday|4.3939393939393945|13.793103448275861| 9.983193277310924|
|             10.05|         20|   Saturday| 5.277777777777778|               0.0|10.746268656716417|
|1.8666666666666667|         20|   Saturday|               7.0|               0.0|16.071428571428573|
|19.316666666666666|         20|   Saturday|               5.0|               0.0| 9.318377911993098|
|20.216666666666665|         20|   Saturday|               3.0|24.814814814814813|26.710634789777412|
|24.866666666666667|         20|   Saturday| 6.363636363636363|               0.0|

We only have 3 rows with null values which is insignificant compared to the remaining dataset

### EXPLORATORY DATA ANALYSIS

Total trips by pickup hour

In [11]:
trips_by_hour = df_clean.groupBy(
    "pickup_hour"
).agg(
    F.count("*").alias("total_trips")
).orderBy(
    "pickup_hour"
)

trips_by_hour.show(24)

+-----------+-----------+
|pickup_hour|total_trips|
+-----------+-----------+
|          0|     858845|
|          1|     649510|
|          2|     492157|
|          3|     362832|
|          4|     264366|
|          5|     235004|
|          6|     499501|
|          7|     855043|
|          8|    1047332|
|          9|    1066000|
|         10|    1042439|
|         11|    1092878|
|         12|    1165840|
|         13|    1160150|
|         14|    1213724|
|         15|    1204321|
|         16|    1079676|
|         17|    1252098|
|         18|    1482999|
|         19|    1479916|
|         20|    1350729|
|         21|    1311984|
|         22|    1260668|
|         23|    1066536|
+-----------+-----------+



Revenue by hour

In [12]:
revenue_by_hour = df_clean.groupBy(
    "pickup_hour"
).agg(
    F.sum("total_amount").alias("total_revenue"),
    F.avg("total_amount").alias("average_trip_revenue")
).orderBy(
    F.desc("total_revenue")
)

revenue_by_hour.show()

+-----------+--------------------+--------------------+
|pickup_hour|       total_revenue|average_trip_revenue|
+-----------+--------------------+--------------------+
|         19|2.5565558970004406E7|   17.27500680444323|
|         18|2.1904747340008218E7|  14.770574585693057|
|         20|2.0218048550005488E7|   14.96824940458485|
|         21|2.0189693740005326E7|   15.38867374907417|
|         22|1.9858303070004858E7|  15.752206822101344|
|         17| 1.921127850000375E7|  15.343270654536427|
|         14|1.8590267890002422E7|  15.316717713419543|
|         15|1.8434007790002387E7|  15.306556798397095|
|         23| 1.726384760000135E7|  16.186840012902845|
|         16|1.7050555720000334E7|  15.792289279376716|
|         13|1.6963267400000136E7|  14.621615653148417|
|         12|1.6585302150000038E7|  14.226053446442082|
|         11|1.5569462019998075E7|  14.246294664178505|
|          9|1.5269917439998131E7|   14.32450041275622|
|          8|1.5026225809997965E7|   14.34714666

Vendor analysis

In [13]:
vendor_analysis = df_clean.groupBy(
    "VendorID"
).agg(
    F.count("*").alias("total_trips"),
    F.avg("trip_distance").alias("average_distance"),
    F.avg("fare_amount").alias("average_fare"),
    F.sum("total_amount").alias("total_revenue")
)

vendor_analysis.show()

+--------+-----------+------------------+------------------+--------------------+
|VendorID|total_trips|  average_distance|      average_fare|       total_revenue|
+--------+-----------+------------------+------------------+--------------------+
|       1|   11073913|16.811235965100906|12.024300006691782|1.6629588405941862E8|
|       2|   12420635| 2.903282545538136|12.233913797482977| 1.934367292693401E8|
+--------+-----------+------------------+------------------+--------------------+



Payment type analysis

In [14]:
payment_analysis = df_clean.groupBy(
    "payment_type"
).agg(
    F.count("*").alias("total_trips"),
    F.avg("tip_amount").alias("average_tip"),
    F.avg("tip_pct").alias("average_tip_percentage"),
    F.sum("total_amount").alias("total_revenue")
).orderBy(
    F.desc("total_revenue")
)

payment_analysis.show()

+------------+-----------+--------------------+----------------------+--------------------+
|payment_type|total_trips|         average_tip|average_tip_percentage|       total_revenue|
+------------+-----------+--------------------+----------------------+--------------------+
|           1|   15005162|   2.821326276917136|    23.315604721888338|2.5355240222963387E8|
|           2|    8414200|1.599165696085189...|  7.468815733684527E-4|1.0503249499946547E8|
|           3|      56393|0.011572890252336283|   0.33623984248629707|   841446.4899999897|
|           4|      18791|0.007437070938215103|   0.02767573709430621|   306251.0099999978|
|           5|          2|                 0.0|                   0.0|                18.6|
+------------+-----------+--------------------+----------------------+--------------------+



### ADVANCED PYSPARK

Approximate Quantiles

In [15]:
distance_quantiles = df_clean.approxQuantile(
    "trip_distance",
    [0.25, 0.50, 0.75, 0.95, 0.99],
    0.01
)

print(distance_quantiles)

[1.0, 1.7, 3.01, 9.6, 15420004.5]


Windows Function

In [16]:
df_clean.createOrReplaceTempView("taxi")
spark_job.sql("""
    SELECT *, RANK() OVER (PARTITION BY pickup_hour ORDER BY fare_amount DESC) AS fare_rank
    FROM taxi
""").filter("fare_rank <= 3").show()

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+------------------+-----------+-----------+------------------+------------------+------------------+---------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|  pickup_longitude|   pickup_latitude|RateCodeID|store_and_fwd_flag| dropoff_longitude|  dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount| trip_duration_min|pickup_hour|day_of_week|     fare_per_mile|           tip_pct| average_speed_mph|fare_rank|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+--

In [17]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy(
    "VendorID"
).orderBy(
    F.col("fare_amount").desc()
)

top_fares = df_clean.withColumn(
    "fare_rank",
    F.rank().over(window_spec)
).filter(
    F.col("fare_rank") <= 3
)

top_fares.select(
    "VendorID",
    "fare_amount",
    "trip_distance",
    "fare_rank"
).show()

+--------+-----------+-------------+---------+
|VendorID|fare_amount|trip_distance|fare_rank|
+--------+-----------+-------------+---------+
|       1|  111270.85|          0.9|        1|
|       1|     8008.0|          2.0|        2|
|       1|     4008.0|          1.7|        3|
|       2|    4001.15|          0.2|        1|
|       2|     3005.5|          0.4|        2|
|       2|      957.6|         0.01|        3|
+--------+-----------+-------------+---------+



Distributed Aggregation

In [18]:
daily_revenue = df_clean.groupBy(
    F.to_date("tpep_pickup_datetime").alias("pickup_date")
).agg(
    F.count("*").alias("total_trips"),
    F.sum("total_amount").alias("daily_revenue")
).orderBy(
    "pickup_date"
)

daily_revenue.show()

+-----------+-----------+------------------+
|pickup_date|total_trips|     daily_revenue|
+-----------+-----------+------------------+
| 2015-01-01|     378888| 5777210.309998332|
| 2015-01-02|     343022| 5078775.609998669|
| 2015-01-03|     404329|  5684695.73999824|
| 2015-01-04|     326468| 5042377.169998785|
| 2015-01-05|     360596| 5462182.849998558|
| 2015-01-06|     381639| 5615890.859998393|
| 2015-01-07|     426745| 6114640.129998136|
| 2015-01-08|     447736| 6536524.339998195|
| 2015-01-09|     444837|  6643962.16999802|
| 2015-01-10|     512041| 7109699.899997952|
| 2015-01-11|     416562|6145285.6899982495|
| 2015-01-12|     393534|  5898301.03999854|
| 2015-01-13|     445445| 6642614.639998085|
| 2015-01-14|     439108|  6626205.60999812|
| 2015-01-15|     447603| 6899103.339998087|
| 2015-01-16|     474516| 7184020.199997896|
| 2015-01-17|     473416| 6583727.899997812|
| 2015-01-18|     423946|  9936724.74999694|
| 2015-01-19|     340268| 5112804.159998787|
| 2015-01-

### FINAL DATA VALIDATION

In [19]:
invalid_records = df_clean.filter(
    (F.col("trip_duration_min") <= 0) |
    (F.col("average_speed_mph") <= 0) |
    (F.col("fare_per_mile") <= 0)
)

print(
    "Invalid processed records:",
    invalid_records.count()
)

Invalid processed records: 0


### Store Processed Data as Parquet

In [20]:
df_clean.write \
    .mode("overwrite") \
    .parquet("output/nyc_taxi_processed")

parquet_taxi_df = spark_job.read.parquet(
    "output/nyc_taxi_processed"
)

parquet_taxi_df.printSchema()

parquet_taxi_df.show(5)

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- day_of_week: string (nu

In [21]:
print(
    "Original processed rows:",
    df_clean.count()
)

print(
    "Parquet rows:",
    parquet_taxi_df.count()
)

Original processed rows: 23494548
Parquet rows: 23494548


In [22]:
parquet_taxi_df = spark_job.read.parquet(
    "output/nyc_taxi_processed"
)

parquet_taxi_df.printSchema()

parquet_taxi_df.show(5, truncate=False)

print("Total rows:", parquet_taxi_df.count())

root
 |-- VendorID: string (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: string (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- day_of_week: string (nu

In [23]:
trips_by_hour = parquet_taxi_df.groupBy(
    "pickup_hour"
).agg(
    F.count("*").alias("total_trips"),
    F.avg("trip_distance").alias("average_trip_distance"),
    F.avg("total_amount").alias("average_trip_amount")
).orderBy(
    "pickup_hour"
)

trips_by_hour.show(24, truncate=False)

+-----------+-----------+---------------------+-------------------+
|pickup_hour|total_trips|average_trip_distance|average_trip_amount|
+-----------+-----------+---------------------+-------------------+
|0          |858845     |3.3079960994125797   |16.0233616543236   |
|1          |649510     |6.409503317885786    |15.661764376225918 |
|2          |492157     |3.2589295692228313   |15.51203491974946  |
|3          |362832     |3.50048733298055     |16.031801109049837 |
|4          |264366     |4.125607982872226    |17.949937170434385 |
|5          |235004     |4.632625444673284    |19.651225255736996 |
|6          |499501     |3.8391936152279897   |15.882226281827704 |
|7          |855043     |2.8484085010929325   |14.499181900800421 |
|8          |1047332    |2.509453611653232    |14.347146664104578 |
|9          |1066000    |2.569825150093816    |14.324500412767813 |
|10         |1042439    |23.294625028419034   |14.344037550408544 |
|11         |1092878    |4.043874485532697    |1

In [24]:
payment_eda = parquet_taxi_df.groupBy(
    "payment_type"
).agg(
    F.count("*").alias("total_trips"),
    F.sum("total_amount").alias("total_revenue"),
    F.avg("total_amount").alias("average_trip_amount"),
    F.avg("tip_amount").alias("average_tip"),
    F.avg("tip_pct").alias("average_tip_percentage")
).orderBy(
    F.desc("total_revenue")
)

payment_eda.show(truncate=False)

+------------+-----------+--------------------+-------------------+--------------------+----------------------+
|payment_type|total_trips|total_revenue       |average_trip_amount|average_tip         |average_tip_percentage|
+------------+-----------+--------------------+-------------------+--------------------+----------------------+
|1           |15005162   |2.535524022315072E8 |16.897678427697564 |2.8213262769178162  |23.315604721890814    |
|2           |8414200    |1.0503249499900027E8|12.48276663247846  |1.599165696085189E-4|7.468815733684529E-4  |
|3           |56393      |841446.4900000495   |14.921115918643261 |0.011572890252336284|0.33623984248629696   |
|4           |18791      |306251.00999999605  |16.297749454525892 |0.007437070938215103|0.02767573709430621   |
|5           |2          |18.6                |9.3                |0.0                 |0.0                   |
+------------+-----------+--------------------+-------------------+--------------------+----------------